# TUM Dataset

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper")

import matplotlib.pyplot as plt

In [ ]:
from pose_estimation import robot_environment_and_headset_data_from_tum, XYZImageGenerationConfig, ICPAlignmentConfig, RobotEnvironment, HeadsetData

tum_rgbd_dataset_location = "../tum_datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = robot_environment_and_headset_data_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
        intervall=(0.1, 0.2)
)

### 3d visualisation

In [ ]:
from pose_estimation import visualize_robot_camera_environment_combo

vis_robot_env, vis_headset, vis_both = False, False, True

if vis_robot_env:
    tum_robot_env.visualize_3d_data()
if vis_headset:
    tum_headset_data.visualize_3d_data()
if vis_both:
    visualize_robot_camera_environment_combo(robot_env=tum_robot_env, headset_data=tum_headset_data)

## Testing Predictors

In [ ]:
from pose_estimation import (
    GradablePosePredictor, OnlyPointsPredictor, ExtractAndMatchWrapperConfig, Augmentation, pose_estimation_ransaac_config_precise,
    EllipsoidPredictor, pose_estimation_ransaac_config_less_precise, PnEDeltaPoseAdamOptimizer
)

In [ ]:
# Creation of the Predictors
no_ellips = GradablePosePredictor(
    creator=OnlyPointsPredictor.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_precise,
        )
    ),
    name="no ellipse"
)

from pose_estimation import SAM3Segmenter, Sam3Prompt, SimpleEllipsoidFitter, MVEEEllipsoidFitter, LeastShellDistanceEllipsoidFitter, GaussianMatchingConfig

s3seg = SAM3Segmenter(Sam3Prompt(text="Object on desk"))
ellips = GradablePosePredictor(
    creator=EllipsoidPredictor.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_precise,
        ),
        cam1_segmenter= s3seg,
        cam2_segmenter= s3seg,
        visualize_segmentation_masks = False,
        ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(contamination=0.4, size_penalty=0.99, size_p_norm=2, visualize=False),
        pne_optimizer = PnEDeltaPoseAdamOptimizer(delta_pose_mapping='euler'),
        matching_config = GaussianMatchingConfig(dummy_value=0.003),
        visualize_pne_optimisation=False,
        visualize_environment_generation= True
    ),
    name="ellipse"
)

In [ ]:
from pose_estimation import NPredictors1DatasetGrader

grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[ellips, no_ellips],
    headset_data = tum_headset_data,
    robot_env = tum_robot_env,
)

In [ ]:
visualize_trajectories_3d = False
if visualize_trajectories_3d:
    grader.visualize_predictions_3d()

grader.print_summary()

grader.print_translational_error_under_limits([0.1, 0.2, 2])

from pose_estimation import TimeSeriesErrorType, SingleValueErrorType

fig1, ax1 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_time_series_error(ax1, TimeSeriesErrorType.ABS_TRANSLATIONAL)

fig2, ax2 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_time_series_error(ax2, TimeSeriesErrorType.ABS_ROTATIONAL)

fig3, ax3 = plt.subplots(1, 1, figsize = (8, 5))
grader.plot_prediction_times(ax3, rotate_x_labels=True)

fig4, ax4 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_time_series_error(ax4, TimeSeriesErrorType.RTE_TRANSLATIONAL)


plt.show()

In [ ]:
from geometric_utilities.slam2mp4 import VideoGenerator

video_predictor = EllipsoidPredictor(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=tum_robot_env.robot_bgr_images,
        cam1_xyz_images=tum_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PyposePNEOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt(text="Items on a desk")),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False
)

init_predictor_grade = PredictionOnDataset(
    predictor = video_predictor,
    headset_data = tum_headset_data,
    number_retry = 1,
    vid_gen=VideoGenerator(fps=20),
    video_save_location="tum.mp4",
    point_cloud=tum_robot_env.robot_xyz_images.reshape(-1,3)
)
init_predictor_grade.print_summary()